# Projeto Prático: Implementação de Banco de Dados Relacional no SQLite

Este notebook documenta a construção, modelagem física e execução de scripts SQL para o sistema de gerenciamento de uma operadora de turismo e viagens. O objetivo principal é garantir a integridade referencial, normalização dos dados e aplicação de boas práticas de Engenharia de Software.

## 1. Escopo e Arquitetura do Ambiente
* **SGBD:** SQLite (`meu_banco.db`)
* **Interface:** Jupyter Notebook com extensão IPython SQL Magic (`%%sql`)
* **ORM/Driver:** SQLAlchemy

In [3]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [ ]:
%sql sqlite:///meu_banco.db

## 2. Ordem de Implementação das Tabelas (DDL)

Para criar as tabelas sem violar as restrições de Chaves Estrangeiras (*Foreign Keys*), a criação seguirá uma ordem lógica de dependência (das tabelas independentes para as tabelas associativas/dependentes):

1. **`CLIENTE`**: Tabela base que armazena os dados dos compradores.
2. **`TELEFONE`**: Entidade dependente de `CLIENTE` (Relacionamento 1:N).
3. **`PACOTE`**: Cadastro dos pacotes de viagem disponíveis.
4. **`GUIA`**: Cadastro dos guias turísticos.
5. **`OFERTA`**: Tabela associativa que conecta `PACOTE` e `GUIA` (Relacionamento N:M).
6. **`VENDA`**: Tabela central que registra as compras, dependendo de `CLIENTE` e `PACOTE`.
7. **`PASSAGEM`**: Entidade dependente de `VENDA` para emissão de bilhetes.

---

## 3. Diretrizes de Implementação no SQLite
* **Ativação de Chaves Estrangeiras:** Por padrão, o SQLite desativa o suporte a *Foreign Keys*. Ativaremos explicitamente usando `PRAGMA foreign_keys = ON;`.
* **Auto-incremento:** Substituição do termo `AUTO_INCREMENT` por `INTEGER PRIMARY KEY AUTOINCREMENT` conforme a sintaxe nativa do motor SQLite.
* **Chaves Compostas:** Implementação das PKs compostas nas tabelas `TELEFONE`, `PASSAGEM` e `OFERTA` usando a restrição `PRIMARY KEY (coluna1, coluna2)` no final do escopo de criação.

Célula 1: Ativação do Suporte a Chaves Estrangeiras
Por padrão, o SQLite não ativa a checagem de integridade referencial automaticamente. Esta linha garante que o banco rejeite inserções inválidas em chaves estrangeiras.

In [4]:
%load_ext sql
%sql sqlite:///meu_banco.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [5]:
%%sql
PRAGMA foreign_keys = ON;

 * sqlite:///meu_banco.db
Done.


[]

Célula 2: Criação das Tabelas Independentes (CLIENTE, PACOTE, GUIA)
Criamos primeiro as entidades que não dependem de nenhuma outra para existir. Note o uso de INTEGER PRIMARY KEY AUTOINCREMENT, que é a forma correta do SQLite gerenciar chaves numéricas sequenciais automaticamente.

In [6]:
%%sql
-- Criação da tabela CLIENTE
CREATE TABLE IF NOT EXISTS CLIENTE (
    Id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
    CPF CHAR(11) UNIQUE NOT NULL,
    Nome VARCHAR(50) NOT NULL,
    Sobrenome VARCHAR(50) NOT NULL
);

-- Criação da tabela PACOTE
CREATE TABLE IF NOT EXISTS PACOTE (
    Id_Pacote INTEGER PRIMARY KEY AUTOINCREMENT,
    Destino VARCHAR(50) NOT NULL,
    Duração INTEGER NOT NULL
);

-- Criação da tabela GUIA
CREATE TABLE IF NOT EXISTS GUIA (
    Id_Guia INTEGER PRIMARY KEY AUTOINCREMENT,
    Nome VARCHAR(50) NOT NULL
);

 * sqlite:///meu_banco.db
Done.
Done.
Done.


[]

Célula 3: Criação da Tabela TELEFONE (Relação 1:N com PK Composta)
Como definido no seu diagrama, esta tabela possui uma chave primária composta pelo ID do cliente e pelo ID do telefone.

In [6]:
%%sql
CREATE TABLE IF NOT EXISTS TELEFONE (
    Id_telefone INTEGER NOT NULL,
    Id_cliente INTEGER NOT NULL,
    Numero CHAR(11) NOT NULL,
    PRIMARY KEY (Id_cliente, Id_telefone),
    FOREIGN KEY (Id_cliente) REFERENCES CLIENTE(Id_cliente) ON DELETE CASCADE
);

 * sqlite:///meu_banco.db
Done.


[]

Célula 5: Criação da Tabela VENDA (Centralizadora)
A tabela de vendas amarra o Id_cliente e o Id_pacote. Ela precisa que ambos existam antes de uma linha ser inserida.

In [7]:
%%sql
CREATE TABLE IF NOT EXISTS VENDA (
    Id_venda INTEGER PRIMARY KEY AUTOINCREMENT,
    Preço DECIMAL(10,2) NOT NULL,
    Data_Compra DATE NOT NULL,
    Id_cliente INTEGER NOT NULL,
    Id_pacote INTEGER NOT NULL,
    FOREIGN KEY (Id_cliente) REFERENCES CLIENTE(Id_cliente),
    FOREIGN KEY (Id_pacote) REFERENCES PACOTE(Id_Pacote)
);

 * sqlite:///meu_banco.db
Done.


[]

Célula 6: Criação da Tabela PASSAGEM (Dependente de Venda)
Por fim, criamos a tabela de passagens que possui uma chave primária composta atrelada à sua respectiva venda.

In [8]:
%%sql
CREATE TABLE IF NOT EXISTS PASSAGEM (
    Id_Passagem INTEGER NOT NULL,
    Id_Venda INTEGER NOT NULL,
    PRIMARY KEY (Id_Venda, Id_Passagem),
    FOREIGN KEY (Id_Venda) REFERENCES VENDA(Id_venda) ON DELETE CASCADE
);

 * sqlite:///meu_banco.db
Done.


[]

Célula 7: Validação da Estrutura
Para garantir que o motor do SQLite interpretou e criou todas as tabelas corretamente no seu arquivo meu_banco.db, execute a query de sistema abaixo:

In [10]:
import sqlite3

# Conecta diretamente ao arquivo binário local no container
conn = sqlite3.connect('meu_banco.db')
cursor = conn.cursor()

# Executa a query de sistema para listar as tabelas criadas
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
tabelas = cursor.fetchall()

# Exibe o resultado de forma limpa no output do Jupyter
print("Tabelas encontradas no banco de dados:")
for tabela in tabelas:
    print(f"- {tabela[0]}")

# Fecha a conexão de validação
conn.close()

Tabelas encontradas no banco de dados:
- CLIENTE
- PACOTE
- GUIA
- TELEFONE
- VENDA
- PASSAGEM


Célula 8: Inserção de Dados (DML)
Copie e cole o bloco abaixo em uma nova célula para popular todas as tabelas com um registro de teste.

In [7]:
%%sql
sqlite:///meu_banco.db

-- Remove todos os registros respeitando a ordem de integridade
DELETE FROM PASSAGEM;
DELETE FROM VENDA;
DELETE FROM TELEFONE;
DELETE FROM CLIENTE;
DELETE FROM PACOTE;
DELETE FROM GUIA;

1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.


[]

In [8]:
%%sql
sqlite:///meu_banco.db

-- 1. Inserir o Cliente (O ID será gerado automaticamente pelo banco)
INSERT INTO CLIENTE (CPF, Nome, Sobrenome) 
VALUES ('12345678901', 'Wallas', 'Oliveira');

-- 2. Inserir o Telefone buscando dinamicamente o ID do cliente recém-criado via CPF
INSERT INTO TELEFONE (Id_telefone, Id_cliente, Numero) 
VALUES (1, (SELECT Id_cliente FROM CLIENTE WHERE CPF = '12345678901'), '77999999999');

-- 3. Inserir o Pacote de Viagem
INSERT INTO PACOTE (Destino, Duração) 
VALUES ('Paris', 7);

-- 4. Inserir o Guia Turístico
INSERT INTO GUIA (Nome) 
VALUES ('Carlos Silva');

-- 5. Registrar a Venda buscando os IDs dinamicamente pelos dados únicos (CPF e Destino)
INSERT INTO VENDA (Preço, Data_Compra, Id_cliente, Id_pacote) 
VALUES (
    4500.00, 
    '2026-05-29', 
    (SELECT Id_cliente FROM CLIENTE WHERE CPF = '12345678901'),
    (SELECT Id_Pacote FROM PACOTE WHERE Destino = 'Paris' ORDER BY Id_Pacote DESC LIMIT 1)
);

-- 6. Emitir a Passagem atrelada à última venda registrada
INSERT INTO PASSAGEM (Id_Passagem, Id_Venda) 
VALUES (1, (SELECT MAX(Id_venda) FROM VENDA));

1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.


[]

Célula 9: Consulta Relacional com JOIN (Ajustada)
Para a demonstração aos alunos, faremos o INNER JOIN conectando o fluxo da venda: partindo da VENDA, descobrimos quem é o CLIENTE (e seu telefone), qual é o PACOTE e qual é a PASSAGEM emitida.

In [9]:
import sqlite3

conn = sqlite3.connect('meu_banco.db')
cursor = conn.cursor()

# Query ajustada conectando apenas as tabelas com chaves validadas no banco
query = """
SELECT 
    C.Nome || ' ' || C.Sobrenome AS Cliente,
    T.Numero AS Telefone,
    P.Destino,
    V.Preço,
    PA.Id_Passagem AS Bilhete
FROM VENDA V
INNER JOIN CLIENTE C ON V.Id_cliente = C.Id_cliente
INNER JOIN TELEFONE T ON C.Id_cliente = T.Id_cliente
INNER JOIN PACOTE P ON V.Id_pacote = P.Id_Pacote
INNER JOIN PASSAGEM PA ON V.Id_venda = PA.Id_Venda;
"""

cursor.execute(query)
resultado = cursor.fetchone()

if resultado:
    print("=== DADOS OPERACIONAIS DA VIAGEM ===")
    print(f"Cliente: {resultado[0]}")
    print(f"Telefone: {resultado[1]}")
    print(f"Destino: {resultado[2]}")
    print(f"Preço do Pacote: R$ {resultado[3]:.2f}")
    print(f"Número do Bilhete (Passagem): {resultado[4]}")
else:
    print("Nenhum registro encontrado.")

conn.close()

=== DADOS OPERACIONAIS DA VIAGEM ===
Cliente: Wallas Oliveira
Telefone: 77999999999
Destino: Paris
Preço do Pacote: R$ 4500.00
Número do Bilhete (Passagem): 1
